# Frequentist Panel Regression: OLS, FEM, REM

**Task:** 13 - Ước lượng OLS, FEM, REM  
**Date:** 2026-03-10  
**Engine:** 
---

## Mô hình

```
GDP_growth_it = α + β_E × E_score_it + β_S × S_score_it + β_G × G_score_it + 
                γ_1 × inflation_it + γ_2 × population_growth_it + u_i + ε_it
```

### Phương pháp ước lượng:
1. **Pooled OLS**: Bỏ qua cấu trúc panel
2. **Fixed Effects Model (FEM)**: Kiểm soát individual effects cố định
3. **Random Effects Model (REM)**: Individual effects ngẫu nhiên
4. **Hausman Test**: So sánh FEM vs REM

---

## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
from linearmodels.panel import PanelOLS, RandomEffects, compare
from linearmodels.panel import compare
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded successfully")

## 2. Load và Preprocessing Data

In [ ]:
# Load data
df = pd.read_csv('./data/processed/asean-esg-final.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nCountries: {df['ISO3 code'].nunique()}")
print(f"Years: {df['year'].min()} - {df['year'].max()}")

In [ ]:
# Preprocessing (tương tự Bayesian notebook)
df_clean = df.copy()

# Điền G_score bằng median theo quốc gia
df_clean['G_score'] = df_clean.groupby('ISO3 code')['G_score'].transform(
    lambda x: x.fillna(x.median())
)
df_clean['G_score'] = df_clean['G_score'].fillna(df_clean['G_score'].median())

# Điền inflation bằng median theo quốc gia
df_clean['inflation'] = df_clean.groupby('ISO3 code')['inflation'].transform(
    lambda x: x.fillna(x.median())
)
df_clean['inflation'] = df_clean['inflation'].fillna(df_clean['inflation'].median())

# Loại bỏ các hàng có missing values ở biến quan trọng
vars_required = ['GDP_growth', 'E_score', 'S_score', 'G_score', 'inflation', 'population_growth']
df_panel = df_clean.dropna(subset=vars_required).copy()

print(f"\nAfter cleaning: {df_panel.shape[0]} observations")
print(f"Missing values:")
print(df_panel[vars_required].isnull().sum())

In [ ]:
# Set multi-index cho panel data
df_panel = df_panel.set_index(['ISO3 code', 'year'])
df_panel = df_panel.sort_index()

print("Panel data structure:")
print(df_panel.head(10))

## 3. Define Variables

In [ ]:
# Biến phụ thuộc
y = df_panel['GDP_growth']

# Biến độc lập
X_vars = ['E_score', 'S_score', 'G_score', 'inflation', 'population_growth']
X = df_panel[X_vars]

# Thêm constant cho OLS
X_with_const = sm.add_constant(X)

print(f"Dependent variable: GDP_growth")
print(f"Independent variables: {X_vars}")
print(f"\nObservations: {len(y)}")
print(f"Entities (countries): {df_panel.index.get_level_values(0).nunique()}")

## 4. Model 1: Pooled OLS

In [ ]:
# Pooled OLS với cluster-robust standard errors
model_ols = OLS(y, X_with_const)
results_ols = model_ols.fit(cov_type='cluster', cov_kwds={'groups': df_panel.index.get_level_values(0)})

print("=" * 80)
print("POOLED OLS RESULTS (Cluster-Robust SE)")
print("=" * 80)
print(results_ols.summary())

## 5. Model 2: Fixed Effects Model (FEM)

In [ ]:
# Fixed Effects Model với entity effects
model_fem = PanelOLS(y, X, entity_effects=True, time_effects=False)
results_fem = model_fem.fit(cov_type='clustered', cluster_entity=True)

print("=" * 80)
print("FIXED EFFECTS MODEL (FEM) RESULTS")
print("=" * 80)
print(results_fem.summary)

## 6. Model 3: Random Effects Model (REM)

In [ ]:
# Random Effects Model
model_rem = RandomEffects(y, X)
results_rem = model_rem.fit(cov_type='clustered', cluster_entity=True)

print("=" * 80)
print("RANDOM EFFECTS MODEL (REM) RESULTS")
print("=" * 80)
print(results_rem.summary)

## 7. Hausman Test: FEM vs REM

In [ ]:
# Hausman Test thủ công
# H0: Random Effects phù hợp hơn (individual effects không tương quan với X)
# H1: Fixed Effects phù hợp hơn (individual effects tương quan với X)

def hausman_test(fe, re):
    """
    Thực hiện Hausman test để so sánh Fixed Effects và Random Effects
    
    Returns:
        chi2: Chi-square statistic
        pval: p-value
        df: degrees of freedom
    """
    # Lấy coefficients (bỏ constant nếu có)
    b_fe = fe.params.values
    b_re = re.params.values
    
    # Lấy variance-covariance matrices
    v_fe = fe.cov.values
    v_re = re.cov.values
    
    # Tính difference
    b_diff = b_fe - b_re
    v_diff = v_fe - v_re
    
    # Hausman statistic
    try:
        chi2 = np.dot(b_diff.T, np.linalg.solve(v_diff, b_diff))
        df = len(b_fe)
        pval = 1 - stats.chi2.cdf(chi2, df)
        return chi2, pval, df
    except np.linalg.LinAlgError:
        print("Warning: Variance matrix is singular, using pseudo-inverse")
        chi2 = np.dot(b_diff.T, np.linalg.pinv(v_diff).dot(b_diff))
        df = len(b_fe)
        pval = 1 - stats.chi2.cdf(chi2, df)
        return chi2, pval, df

# Thực hiện Hausman test
chi2, pval, df = hausman_test(results_fem, results_rem)

print("=" * 80)
print("HAUSMAN TEST: Fixed Effects vs Random Effects")
print("=" * 80)
print(f"\nChi-square statistic: {chi2:.4f}")
print(f"Degrees of freedom: {df}")
print(f"P-value: {pval:.4f}")
print(f"\nKết luận:")
if pval < 0.05:
    print("  → Bác bỏ H0 (p < 0.05)")
    print("  → Fixed Effects Model (FEM) phù hợp hơn")
    print("  → Individual effects CÓ tương quan với các biến độc lập")
else:
    print("  → Không bác bỏ H0 (p >= 0.05)")
    print("  → Random Effects Model (REM) phù hợp hơn")
    print("  → Individual effects KHÔNG tương quan với các biến độc lập")
print("=" * 80)

## 8. Model Comparison Summary

In [ ]:
# Tạo bảng so sánh các mô hình
comparison_data = []

# Pooled OLS
comparison_data.append({
    'Model': 'Pooled OLS',
    'R-squared': f"{results_ols.rsquared:.4f}",
    'Adj. R-squared': f"{results_ols.rsquared_adj:.4f}",
    'N': int(results_ols.nobs),
    'Entities': df_panel.index.get_level_values(0).nunique()
})

# FEM
comparison_data.append({
    'Model': 'Fixed Effects',
    'R-squared': f"{results_fem.rsquared_within:.4f}",
    'Adj. R-squared': '-',
    'N': results_fem.nobs,
    'Entities': results_fem.entity_info['total']
})

# REM
comparison_data.append({
    'Model': 'Random Effects',
    'R-squared': f"{results_rem.rsquared:.4f}",
    'Adj. R-squared': '-',
    'N': results_rem.nobs,
    'Entities': results_rem.entity_info['total']
})

df_comparison = pd.DataFrame(comparison_data)
print("\n" + "=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)
print(df_comparison.to_string(index=False))

## 9. Extract Results for LaTeX Table

In [ ]:
# Hàm format số với chuẩn mực học thuật
def format_coef(coef, se, pval, stars=True):
    """Format coefficient với standard error trong ngoặc"""
    if stars:
        if pval < 0.01:
            star = '***'
        elif pval < 0.05:
            star = '**'
        elif pval < 0.10:
            star = '*'
        else:
            star = ''
        return f"{coef:.4f}{star}", f"({se:.4f})"
    else:
        return f"{coef:.4f}", f"({se:.4f})"

# Tạo dictionary để lưu kết quả
results_dict = {
    'Variable': [],
    'OLS_coef': [], 'OLS_se': [],
    'FEM_coef': [], 'FEM_se': [],
    'REM_coef': [], 'REM_se': []
}

for var in X_vars:
    results_dict['Variable'].append(var)
    
    # OLS
    ols_coef, ols_se = format_coef(
        results_ols.params[var],
        results_ols.bse[var],
        results_ols.pvalues[var]
    )
    results_dict['OLS_coef'].append(ols_coef)
    results_dict['OLS_se'].append(ols_se)
    
    # FEM
    fem_coef, fem_se = format_coef(
        results_fem.params[var],
        results_fem.std_errors[var],
        results_fem.pvalues[var]
    )
    results_dict['FEM_coef'].append(fem_coef)
    results_dict['FEM_se'].append(fem_se)
    
    # REM
    rem_coef, rem_se = format_coef(
        results_rem.params[var],
        results_rem.std_errors[var],
        results_rem.pvalues[var]
    )
    results_dict['REM_coef'].append(rem_coef)
    results_dict['REM_se'].append(rem_se)

df_results = pd.DataFrame(results_dict)
print("\n" + "=" * 80)
print("COEFFICIENT TABLE (for LaTeX)")
print("=" * 80)
print(df_results.to_string(index=False))

In [ ]:
# Lưu kết quả để sử dụng cho LaTeX table
output_data = {
    'variables': X_vars,
    'ols': {
        'params': results_ols.params[X_vars].to_dict(),
        'se': results_ols.bse[X_vars].to_dict(),
        'pvalues': results_ols.pvalues[X_vars].to_dict(),
        'r2': results_ols.rsquared,
        'n': int(results_ols.nobs)
    },
    'fem': {
        'params': results_fem.params.to_dict(),
        'se': results_fem.std_errors.to_dict(),
        'pvalues': results_fem.pvalues.to_dict(),
        'r2_within': results_fem.rsquared_within,
        'n': results_fem.nobs
    },
    'rem': {
        'params': results_rem.params.to_dict(),
        'se': results_rem.std_errors.to_dict(),
        'pvalues': results_rem.pvalues.to_dict(),
        'r2': results_rem.rsquared,
        'n': results_rem.nobs
    },
    'hausman': {
        'chi2': chi2,
        'pval': pval,
        'df': df
    }
}

import json
with open('./data/processed/frequentist-results.json', 'w') as f:
    json.dump(output_data, f, indent=2)

print("\n✓ Results saved to: data/processed/frequentist-results.json")

## 10. Kết luận

In [ ]:
print("\n" + "=" * 80)
print("KẾT LUẬN")
print("=" * 80)

print("\n1. HAUSMAN TEST:")
print(f"   Chi-square = {chi2:.4f}, p-value = {pval:.4f}")
if pval < 0.05:
    print("   → Ưu tiên sử dụng FIXED EFFECTS MODEL")
    best_model = 'FEM'
else:
    print("   → Ưu tiên sử dụng RANDOM EFFECTS MODEL")
    best_model = 'REM'

print(f"\n2. MÔ HÌNH TỐI ƯU: {best_model}")

print("\n3. KẾT QUẢ CHÍNH:")
if best_model == 'FEM':
    for var in X_vars:
        coef = results_fem.params[var]
        pval = results_fem.pvalues[var]
        sig = '***' if pval < 0.01 else ('**' if pval < 0.05 else ('*' if pval < 0.10 else ''))
        print(f"   {var}: {coef:.4f}{sig} (p={pval:.4f})")
else:
    for var in X_vars:
        coef = results_rem.params[var]
        pval = results_rem.pvalues[var]
        sig = '***' if pval < 0.01 else ('**' if pval < 0.05 else ('*' if pval < 0.10 else ''))
        print(f"   {var}: {coef:.4f}{sig} (p={pval:.4f})")

print("\n" + "=" * 80)